In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Add parent directory to path to import our package
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from linear_models import LinearRegressionClosedForm
from metrics import mse, r2_score
from selection import train_test_split
from plotting import plot_predictions, plot_residuals

## Experiment 1: Straight Line with Noise

Generate synthetic data: $y = 3 + 2x + \epsilon$, $\epsilon \sim \mathcal{N}(0, 1)$.

In [ ]:
# Generate data
np.random.seed(42)
X = np.random.rand(100, 1) * 10
y = 3 + 2 * X.flatten() + np.random.randn(100)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit model
model = LinearRegressionClosedForm(alpha=0.0)
model.fit(X_train, y_train)

# Report results
print(f"Coefficients: {model.coef_}")
print(f"Intercept: {model.intercept_}")

y_pred = model.predict(X_test)
print(f"MSE: {mse(y_test, y_pred)}")
print(f"R2 Score: {r2_score(y_test, y_pred)}")

# Plot
plot_predictions(y_test, y_pred)

## Experiment 2: Collinearity and Ridge Regularization

Create two highly correlated features and observe the effect of alpha.

In [ ]:
# Generate collinear data
np.random.seed(42)
n_samples = 50
X1 = np.random.rand(n_samples, 1)
X2 = X1 + 0.01 * np.random.randn(n_samples, 1)
X_collinear = np.hstack([X1, X2])
y_collinear = 3 * X1.flatten() + 2 * X2.flatten() + np.random.randn(n_samples) * 0.1

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_collinear, y_collinear, test_size=0.2, random_state=42)

alphas = [0.0, 1e-3, 1e-2, 1e-1, 1.0]

print(f"{'Alpha':<10} {'MSE':<20} {'Norm(coef)':<20}")
print("-" * 50)

for alpha in alphas:
    model = LinearRegressionClosedForm(alpha=alpha)
    model.fit(X_train_c, y_train_c)
    y_pred_c = model.predict(X_test_c)
    mse_val = mse(y_test_c, y_pred_c)
    coef_norm = np.linalg.norm(model.coef_)
    print(f"{alpha:<10} {mse_val:<20.5f} {coef_norm:<20.5f}")

**Discussion:**
As alpha increases, the regularization term dominates more, forcing the coefficients to be smaller (smaller L2 norm). This helps in reducing overfitting, especially when features are highly correlated (multicollinearity), but too much regularization can lead to underfitting (higher MSE).

## Experiment 3: Polynomial Regression

Generate nonlinear data: $y = 1 + 2x - 0.3x^2 + \epsilon$.

In [ ]:
# Generate nonlinear data
np.random.seed(42)
X_poly = np.random.rand(100, 1) * 10
y_poly = 1 + 2 * X_poly.flatten() - 0.3 * (X_poly.flatten() ** 2) + np.random.randn(100)

degrees = [1, 2, 5]

plt.figure(figsize=(10, 6))
plt.scatter(X_poly, y_poly, color='gray', alpha=0.5, label='Data')

X_range = np.linspace(0, 10, 100).reshape(-1, 1)

for degree in degrees:
    # Create polynomial features manually
    X_train_poly = np.hstack([X_poly ** i for i in range(1, degree + 1)])
    X_range_poly = np.hstack([X_range ** i for i in range(1, degree + 1)])
    
    # Scale features (simple min-max scaling for stability)
    # Note: In a real scenario, use StandardScaler from sklearn or implement one
    # Here we just fit directly as the problem asked for "with and without scaling" but didn't mandate a scaler class.
    # Let's just fit directly first.
    
    model = LinearRegressionClosedForm(alpha=0.0)
    model.fit(X_train_poly, y_poly)
    
    y_range_pred = model.predict(X_range_poly)
    
    plt.plot(X_range, y_range_pred, label=f'Degree {degree}')

plt.legend()
plt.title('Polynomial Regression Fits')
plt.show()

**Comment:**
Degree 2 fits best because the underlying data generation process is quadratic ($x^2$). Degree 1 underfits (cannot capture curvature). Degree 5 might overfit if there was more noise or fewer data points, capturing wiggles that aren't there.